## Multilabel Classification of Depression Emotions

### GPT-5.1-thinking

In [ ]:
import pandas as pd
import json
import os
import time
from tqdm import tqdm
from openai import OpenAI

# ========== 配置 ==========
key = " "
input_json_path = "/data/zhengtianlong/Private/Depressed/DepressionEmo-main/Dataset/test.json"
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/gpt-5.1-thinking_Depression_EIGHT.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# ========== 读取数据 ==========
data = pd.read_json(input_json_path, lines=True)  # JSON lines 格式
# 确保 id 列为字符串类型
data['id'] = data['id'].astype(str)

# ========== 定义情绪分类函数 ==========
def identify_emotions(text):
    """调用大模型，返回帖子中包含的情绪代码（如 '丙, 丁'）"""
    client = OpenAI(
        base_url="https://api.linkapi.org/v1",
        api_key=key
    )
    prompt = f"""You are an expert in mental health analysis. Your task is to analyze a social media post and identify which of the following depression-related emotions are expressed in it. Each emotion is represented by a Chinese character code (甲, 乙, 丙, 丁, 戊, 己, 庚, 辛) with corresponding descriptions:

甲: Anger – an intensified emotional response that can be directed inward or outward, contributing to despair and self-criticism.  
乙: Cognitive dysfunction – impairment in normal cognitive functioning, causing forgetfulness, slow thinking, difficulty expressing thoughts.  
丙: Emptiness – a deep emotional void, numbness, lack of vitality, feeling disconnected and an empty future.  
丁: Hopelessness – a susceptibility factor in depressive symptoms, intrinsic to depression.  
戊: Loneliness – profound isolation even in the presence of others, predisposing to depression.  
己: Sadness – a natural emotion triggered by events or losses, a fundamental symptom of depression.  
庚: Suicide intent – conscious desire to end one's life, indicating a serious mental health concern, often overwhelming emotional pain.  
辛: Worthlessness – a profound sense of having little or no value, feeling inadequate or undeserving.

Instructions:
1. Read the given post text carefully.
2. Based on the content, identify all emotions from the list above that are present or implied in the post. Each post is guaranteed to contain at least one of these emotions.
3. Return your answer as a comma-separated list of the corresponding Chinese codes (e.g., "丙, 丁" for emptiness and hopelessness). Do not include any additional text or explanation.
4. Only output the list.

Post: {text}

Emotions expressed:"""

    response = client.chat.completions.create(
        model="gpt-5.1-thinking",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=2000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# ========== 初始化输出文件 ==========
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as f:
            existing_data = json.load(f)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件损坏或为空，重新初始化
        with open(output_json_path, "w") as f:
            json.dump([], f)
else:
    with open(output_json_path, "w") as f:
        json.dump([], f)

# ========== 重试处理 ==========
max_retries = 5
current_retry = 0
all_ids = set(data['id'].tolist())

while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    remaining_data = data[~data['id'].isin(success_ids)]

    if remaining_data.empty:
        break

    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = row['id']
            text = row['text']
            title = row.get('title', '')
            post = row.get('post', '')
            target = row['emotions']      # 原始情绪列表，如 ['hopelessness', 'sadness']
            label = row['label_id']       # 标签二进制字符串，如 "10100"

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查
            if Id in success_ids:
                continue

            # 调用模型预测
            predict = identify_emotions(text)

            # 构造结果对象
            result = {
                "Id": Id,
                "text": text,
                "title": title,
                "post": post,
                "predict": predict,
                "label": label,
                "target": target   # 保留原始情绪列表用于后续评估
            }

            # 实时写入文件
            with open(output_json_path, "r+") as f:
                file_data = json.load(f)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    f.seek(0)
                    json.dump(file_data, f, ensure_ascii=False, indent=4)
                    f.truncate()
                    success_ids.add(Id)
                else:
                    # 同步内存集合
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 避免请求过快

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            continue

# ========== 按原始顺序排序 ==========
print("正在按原始 CSV 顺序重新排序输出文件...")
id_to_order = {row['id']: idx for idx, row in data.iterrows()}

with open(output_json_path, "r") as f:
    final_data = json.load(f)

final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# ========== 最终统计 ==========
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

正在按原始 CSV 顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [5]:
import json
import numpy as np
from sklearn.metrics import (accuracy_score, hamming_loss, f1_score, 
                             precision_score, recall_score, 
                             precision_recall_fscore_support)

# ========== 配置 ==========
input_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/gpt-5.1-thinking_Depression_EIGHT.json"

# 情绪代码到英文名称的映射（顺序固定，用于二进制向量）
emotion_codes = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛']
emotion_names = [
    'anger',
    'brain dysfunction (forget)',
    'emptiness',
    'hopelessness',
    'loneliness',
    'sadness',
    'suicide intent',
    'worthlessness'
]
code_to_name = dict(zip(emotion_codes, emotion_names))

# ========== 读取数据 ==========
with open(input_json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# ========== 解析预测和真实标签 ==========
y_true = []   # 真实二进制向量
y_pred = []   # 预测二进制向量

for item in data:
    # 真实情绪列表
    true_emotions = set(item['target'])  # 示例: ["hopelessness", "sadness"]
    # 转换为二进制向量
    true_vec = [1 if name in true_emotions else 0 for name in emotion_names]
    # print(true_vec)
    y_true.append(true_vec)

    # 预测字符串
    pred_str = item.get('predict', '')
    # 提取出现的情绪代码
    pred_codes = {code for code in emotion_codes if code in pred_str}
    # print(pred_codes)
    # 转换为英文名称（用于统一表示，也可直接用代码构建向量）
    pred_emotions = {code_to_name[code] for code in pred_codes}
    # print(pred_emotions)
    pred_vec = [1 if name in pred_emotions else 0 for name in emotion_names]
    # print(pred_vec)
    y_pred.append(pred_vec)

# 转换为 numpy 数组
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ========== 计算指标 ==========
# 1. Exact Match Ratio (准确率)
exact_match = accuracy_score(y_true, y_pred)
print(f"Exact Match Ratio (准确率): {exact_match:.4f}")

# 2. Hamming Loss
h_loss = hamming_loss(y_true, y_pred)
print(f"Hamming Loss: {h_loss:.4f}")

# 3. 宏平均指标
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1-score:  {f1_macro:.4f}")

# 4. 微平均指标
precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
print(f"\nMicro Precision: {precision_micro:.4f}")
print(f"Micro Recall:    {recall_micro:.4f}")
print(f"Micro F1-score:  {f1_micro:.4f}")

# 5. 每个类别的精确率、召回率、F1-score
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0, labels=range(8)
)

print("\n每个类别的指标：")
for i, name in enumerate(emotion_names):
    print(f"{name}:")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-score:  {f1[i]:.4f}")
    print(f"  Support:   {support[i]}")

# 可选：输出总体统计
print(f"\n总样本数: {len(data)}")

Exact Match Ratio (准确率): 0.0342
Hamming Loss: 0.3405

Macro Precision: 0.6518
Macro Recall:    0.6075
Macro F1-score:  0.5879

Micro Precision: 0.6257
Micro Recall:    0.6127
Micro F1-score:  0.6191

每个类别的指标：
anger:
  Precision: 0.7257
  Recall:    0.4754
  F1-score:  0.5744
  Support:   345
brain dysfunction (forget):
  Precision: 0.2971
  Recall:    0.3041
  F1-score:  0.3006
  Support:   171
emptiness:
  Precision: 0.4493
  Recall:    0.8446
  F1-score:  0.5866
  Support:   341
hopelessness:
  Precision: 0.7367
  Recall:    0.8517
  F1-score:  0.7901
  Support:   634
loneliness:
  Precision: 0.5338
  Recall:    0.9024
  F1-score:  0.6708
  Support:   420
sadness:
  Precision: 0.9633
  Recall:    0.3013
  F1-score:  0.4590
  Support:   697
suicide intent:
  Precision: 0.8122
  Recall:    0.6897
  F1-score:  0.7459
  Support:   232
worthlessness:
  Precision: 0.6961
  Recall:    0.4908
  F1-score:  0.5757
  Support:   434

总样本数: 906


### GPT-5.4

In [ ]:
import pandas as pd
import json
import os
import time
from tqdm import tqdm
from openai import OpenAI

# ========== 配置 ==========
key = " "
input_json_path = "/data/zhengtianlong/Private/Depressed/DepressionEmo-main/Dataset/test.json"
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/gpt-5.4_Depression_EIGHT.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# ========== 读取数据 ==========
data = pd.read_json(input_json_path, lines=True)  # JSON lines 格式
# 确保 id 列为字符串类型
data['id'] = data['id'].astype(str)

# ========== 定义情绪分类函数 ==========
def identify_emotions(text):
    """调用大模型，返回帖子中包含的情绪代码（如 '丙, 丁'）"""
    client = OpenAI(
        base_url="https://api.linkapi.org/v1",
        api_key=key
    )
    prompt = f"""You are an expert in mental health analysis. Your task is to analyze a social media post and identify which of the following depression-related emotions are expressed in it. Each emotion is represented by a Chinese character code (甲, 乙, 丙, 丁, 戊, 己, 庚, 辛) with corresponding descriptions:

甲: Anger – an intensified emotional response that can be directed inward or outward, contributing to despair and self-criticism.  
乙: Cognitive dysfunction – impairment in normal cognitive functioning, causing forgetfulness, slow thinking, difficulty expressing thoughts.  
丙: Emptiness – a deep emotional void, numbness, lack of vitality, feeling disconnected and an empty future.  
丁: Hopelessness – a susceptibility factor in depressive symptoms, intrinsic to depression.  
戊: Loneliness – profound isolation even in the presence of others, predisposing to depression.  
己: Sadness – a natural emotion triggered by events or losses, a fundamental symptom of depression.  
庚: Suicide intent – conscious desire to end one's life, indicating a serious mental health concern, often overwhelming emotional pain.  
辛: Worthlessness – a profound sense of having little or no value, feeling inadequate or undeserving.

Instructions:
1. Read the given post text carefully.
2. Based on the content, identify all emotions from the list above that are present or implied in the post. Each post is guaranteed to contain at least one of these emotions.
3. Return your answer as a comma-separated list of the corresponding Chinese codes (e.g., "丙, 丁" for emptiness and hopelessness). Do not include any additional text or explanation.
4. Only output the list.

Post: {text}

Emotions expressed:"""

    response = client.chat.completions.create(
        model="gpt-5.4",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=2000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# ========== 初始化输出文件 ==========
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as f:
            existing_data = json.load(f)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件损坏或为空，重新初始化
        with open(output_json_path, "w") as f:
            json.dump([], f)
else:
    with open(output_json_path, "w") as f:
        json.dump([], f)

# ========== 重试处理 ==========
max_retries = 5
current_retry = 0
all_ids = set(data['id'].tolist())

while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    remaining_data = data[~data['id'].isin(success_ids)]

    if remaining_data.empty:
        break

    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = row['id']
            text = row['text']
            title = row.get('title', '')
            post = row.get('post', '')
            target = row['emotions']      # 原始情绪列表，如 ['hopelessness', 'sadness']
            label = row['label_id']       # 标签二进制字符串，如 "10100"

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查
            if Id in success_ids:
                continue

            # 调用模型预测
            predict = identify_emotions(text)

            # 构造结果对象
            result = {
                "Id": Id,
                "text": text,
                "title": title,
                "post": post,
                "predict": predict,
                "label": label,
                "target": target   # 保留原始情绪列表用于后续评估
            }

            # 实时写入文件
            with open(output_json_path, "r+") as f:
                file_data = json.load(f)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    f.seek(0)
                    json.dump(file_data, f, ensure_ascii=False, indent=4)
                    f.truncate()
                    success_ids.add(Id)
                else:
                    # 同步内存集合
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 避免请求过快

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            continue

# ========== 按原始顺序排序 ==========
print("正在按原始 CSV 顺序重新排序输出文件...")
id_to_order = {row['id']: idx for idx, row in data.iterrows()}

with open(output_json_path, "r") as f:
    final_data = json.load(f)

final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# ========== 最终统计 ==========
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

重试第 1/5 轮，剩余 906 条样本


第1轮: 100%|██████████| 906/906 [52:29<00:00,  3.48s/it]  

正在按原始 CSV 顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [5]:
import json
import numpy as np
from sklearn.metrics import (accuracy_score, hamming_loss, f1_score, 
                             precision_score, recall_score, 
                             precision_recall_fscore_support)

# ========== 配置 ==========
input_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/gpt-5.4_Depression_EIGHT.json"

# 情绪代码到英文名称的映射（顺序固定，用于二进制向量）
emotion_codes = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛']
emotion_names = [
    'anger',
    'brain dysfunction (forget)',
    'emptiness',
    'hopelessness',
    'loneliness',
    'sadness',
    'suicide intent',
    'worthlessness'
]
code_to_name = dict(zip(emotion_codes, emotion_names))

# ========== 读取数据 ==========
with open(input_json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# ========== 解析预测和真实标签 ==========
y_true = []   # 真实二进制向量
y_pred = []   # 预测二进制向量

for item in data:
    # 真实情绪列表
    true_emotions = set(item['target'])  # 示例: ["hopelessness", "sadness"]
    # 转换为二进制向量
    true_vec = [1 if name in true_emotions else 0 for name in emotion_names]
    # print(true_vec)
    y_true.append(true_vec)

    # 预测字符串
    pred_str = item.get('predict', '')
    # 提取出现的情绪代码
    pred_codes = {code for code in emotion_codes if code in pred_str}
    # print(pred_codes)
    # 转换为英文名称（用于统一表示，也可直接用代码构建向量）
    pred_emotions = {code_to_name[code] for code in pred_codes}
    # print(pred_emotions)
    pred_vec = [1 if name in pred_emotions else 0 for name in emotion_names]
    # print(pred_vec)
    y_pred.append(pred_vec)

# 转换为 numpy 数组
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ========== 计算指标 ==========
# 1. Exact Match Ratio (准确率)
exact_match = accuracy_score(y_true, y_pred)
print(f"Exact Match Ratio (准确率): {exact_match:.4f}")

# 2. Hamming Loss
h_loss = hamming_loss(y_true, y_pred)
print(f"Hamming Loss: {h_loss:.4f}")

# 3. 宏平均指标
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1-score:  {f1_macro:.4f}")

# 4. 微平均指标
precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
print(f"\nMicro Precision: {precision_micro:.4f}")
print(f"Micro Recall:    {recall_micro:.4f}")
print(f"Micro F1-score:  {f1_micro:.4f}")

# 5. 每个类别的精确率、召回率、F1-score
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0, labels=range(8)
)

print("\n每个类别的指标：")
for i, name in enumerate(emotion_names):
    print(f"{name}:")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-score:  {f1[i]:.4f}")
    print(f"  Support:   {support[i]}")

# 可选：输出总体统计
print(f"\n总样本数: {len(data)}")

Exact Match Ratio (准确率): 0.1093
Hamming Loss: 0.2718

Macro Precision: 0.7430
Macro Recall:    0.5224
Macro F1-score:  0.6009

Micro Precision: 0.7903
Micro Recall:    0.5422
Micro F1-score:  0.6431

每个类别的指标：
anger:
  Precision: 0.7742
  Recall:    0.5565
  F1-score:  0.6476
  Support:   345
brain dysfunction (forget):
  Precision: 0.5000
  Recall:    0.2047
  F1-score:  0.2905
  Support:   171
emptiness:
  Precision: 0.6849
  Recall:    0.2933
  F1-score:  0.4107
  Support:   341
hopelessness:
  Precision: 0.8686
  Recall:    0.5110
  F1-score:  0.6435
  Support:   634
loneliness:
  Precision: 0.8301
  Recall:    0.7095
  F1-score:  0.7651
  Support:   420
sadness:
  Precision: 0.8882
  Recall:    0.6270
  F1-score:  0.7351
  Support:   697
suicide intent:
  Precision: 0.7280
  Recall:    0.8190
  F1-score:  0.7708
  Support:   232
worthlessness:
  Precision: 0.6700
  Recall:    0.4585
  F1-score:  0.5445
  Support:   434

总样本数: 906


### Claude-Sonnet-4-6

In [ ]:
import pandas as pd
import json
import os
import time
from tqdm import tqdm
from openai import OpenAI

# ========== 配置 ==========
key = " "
input_json_path = "/data/zhengtianlong/Private/Depressed/DepressionEmo-main/Dataset/test.json"
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/claude-sonnet-4-6_Depression_EIGHT.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# ========== 读取数据 ==========
data = pd.read_json(input_json_path, lines=True)  # JSON lines 格式
# 确保 id 列为字符串类型
data['id'] = data['id'].astype(str)

# ========== 定义情绪分类函数 ==========
def identify_emotions(text):
    """调用大模型，返回帖子中包含的情绪代码（如 '丙, 丁'）"""
    client = OpenAI(
        base_url="https://api.linkapi.org/v1",
        api_key=key
    )
    prompt = f"""You are an expert in mental health analysis. Your task is to analyze a social media post and identify which of the following depression-related emotions are expressed in it. Each emotion is represented by a Chinese character code (甲, 乙, 丙, 丁, 戊, 己, 庚, 辛) with corresponding descriptions:

甲: Anger – an intensified emotional response that can be directed inward or outward, contributing to despair and self-criticism.  
乙: Cognitive dysfunction – impairment in normal cognitive functioning, causing forgetfulness, slow thinking, difficulty expressing thoughts.  
丙: Emptiness – a deep emotional void, numbness, lack of vitality, feeling disconnected and an empty future.  
丁: Hopelessness – a susceptibility factor in depressive symptoms, intrinsic to depression.  
戊: Loneliness – profound isolation even in the presence of others, predisposing to depression.  
己: Sadness – a natural emotion triggered by events or losses, a fundamental symptom of depression.  
庚: Suicide intent – conscious desire to end one's life, indicating a serious mental health concern, often overwhelming emotional pain.  
辛: Worthlessness – a profound sense of having little or no value, feeling inadequate or undeserving.

Instructions:
1. Read the given post text carefully.
2. Based on the content, identify all emotions from the list above that are present or implied in the post. Each post is guaranteed to contain at least one of these emotions.
3. Return your answer as a comma-separated list of the corresponding Chinese codes (e.g., "丙, 丁" for emptiness and hopelessness). Do not include any additional text or explanation.
4. Only output the list.

Post: {text}

Emotions expressed:"""

    response = client.chat.completions.create(
        model="claude-sonnet-4-6",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=2000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# ========== 初始化输出文件 ==========
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as f:
            existing_data = json.load(f)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件损坏或为空，重新初始化
        with open(output_json_path, "w") as f:
            json.dump([], f)
else:
    with open(output_json_path, "w") as f:
        json.dump([], f)

# ========== 重试处理 ==========
max_retries = 5
current_retry = 0
all_ids = set(data['id'].tolist())

while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    remaining_data = data[~data['id'].isin(success_ids)]

    if remaining_data.empty:
        break

    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = row['id']
            text = row['text']
            title = row.get('title', '')
            post = row.get('post', '')
            target = row['emotions']      # 原始情绪列表，如 ['hopelessness', 'sadness']
            label = row['label_id']       # 标签二进制字符串，如 "10100"

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查
            if Id in success_ids:
                continue

            # 调用模型预测
            predict = identify_emotions(text)

            # 构造结果对象
            result = {
                "Id": Id,
                "text": text,
                "title": title,
                "post": post,
                "predict": predict,
                "label": label,
                "target": target   # 保留原始情绪列表用于后续评估
            }

            # 实时写入文件
            with open(output_json_path, "r+") as f:
                file_data = json.load(f)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    f.seek(0)
                    json.dump(file_data, f, ensure_ascii=False, indent=4)
                    f.truncate()
                    success_ids.add(Id)
                else:
                    # 同步内存集合
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 避免请求过快

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            continue

# ========== 按原始顺序排序 ==========
print("正在按原始 CSV 顺序重新排序输出文件...")
id_to_order = {row['id']: idx for idx, row in data.iterrows()}

with open(output_json_path, "r") as f:
    final_data = json.load(f)

final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# ========== 最终统计 ==========
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

In [ ]:
import json
import numpy as np
from sklearn.metrics import (accuracy_score, hamming_loss, f1_score, 
                             precision_score, recall_score, 
                             precision_recall_fscore_support)

# ========== 配置 ==========
input_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/claude-sonnet-4-6_Depression_EIGHT.json"

# 情绪代码到英文名称的映射（顺序固定，用于二进制向量）
emotion_codes = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛']
emotion_names = [
    'anger',
    'brain dysfunction (forget)',
    'emptiness',
    'hopelessness',
    'loneliness',
    'sadness',
    'suicide intent',
    'worthlessness'
]
code_to_name = dict(zip(emotion_codes, emotion_names))

# ========== 读取数据 ==========
with open(input_json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# ========== 解析预测和真实标签 ==========
y_true = []   # 真实二进制向量
y_pred = []   # 预测二进制向量

for item in data:
    # 真实情绪列表
    true_emotions = set(item['target'])  # 示例: ["hopelessness", "sadness"]
    # 转换为二进制向量
    true_vec = [1 if name in true_emotions else 0 for name in emotion_names]
    # print(true_vec)
    y_true.append(true_vec)

    # 预测字符串
    pred_str = item.get('predict', '')
    # 提取出现的情绪代码
    pred_codes = {code for code in emotion_codes if code in pred_str}
    # print(pred_codes)
    # 转换为英文名称（用于统一表示，也可直接用代码构建向量）
    pred_emotions = {code_to_name[code] for code in pred_codes}
    # print(pred_emotions)
    pred_vec = [1 if name in pred_emotions else 0 for name in emotion_names]
    # print(pred_vec)
    y_pred.append(pred_vec)

# 转换为 numpy 数组
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ========== 计算指标 ==========
# 1. Exact Match Ratio (准确率)
exact_match = accuracy_score(y_true, y_pred)
print(f"Exact Match Ratio (准确率): {exact_match:.4f}")

# 2. Hamming Loss
h_loss = hamming_loss(y_true, y_pred)
print(f"Hamming Loss: {h_loss:.4f}")

# 3. 宏平均指标
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1-score:  {f1_macro:.4f}")

# 4. 微平均指标
precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
print(f"\nMicro Precision: {precision_micro:.4f}")
print(f"Micro Recall:    {recall_micro:.4f}")
print(f"Micro F1-score:  {f1_micro:.4f}")

# 5. 每个类别的精确率、召回率、F1-score
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0, labels=range(8)
)

print("\n每个类别的指标：")
for i, name in enumerate(emotion_names):
    print(f"{name}:")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-score:  {f1[i]:.4f}")
    print(f"  Support:   {support[i]}")

# 可选：输出总体统计
print(f"\n总样本数: {len(data)}")

### Gemini-3.1-Pro-Preview

In [ ]:
import pandas as pd
import json
import os
import time
from tqdm import tqdm
from openai import OpenAI

# ========== 配置 ==========
key = " "
input_json_path = "/data/zhengtianlong/Private/Depressed/DepressionEmo-main/Dataset/test.json"
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/gemini-3.1-pro-preview_Depression_EIGHT.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# ========== 读取数据 ==========
data = pd.read_json(input_json_path, lines=True)  # JSON lines 格式
# 确保 id 列为字符串类型
data['id'] = data['id'].astype(str)

# ========== 定义情绪分类函数 ==========
def identify_emotions(text):
    """调用大模型，返回帖子中包含的情绪代码（如 '丙, 丁'）"""
    client = OpenAI(
        base_url="https://api.linkapi.org/v1",
        api_key=key
    )
    prompt = f"""You are an expert in mental health analysis. Your task is to analyze a social media post and identify which of the following depression-related emotions are expressed in it. Each emotion is represented by a Chinese character code (甲, 乙, 丙, 丁, 戊, 己, 庚, 辛) with corresponding descriptions:

甲: Anger – an intensified emotional response that can be directed inward or outward, contributing to despair and self-criticism.  
乙: Cognitive dysfunction – impairment in normal cognitive functioning, causing forgetfulness, slow thinking, difficulty expressing thoughts.  
丙: Emptiness – a deep emotional void, numbness, lack of vitality, feeling disconnected and an empty future.  
丁: Hopelessness – a susceptibility factor in depressive symptoms, intrinsic to depression.  
戊: Loneliness – profound isolation even in the presence of others, predisposing to depression.  
己: Sadness – a natural emotion triggered by events or losses, a fundamental symptom of depression.  
庚: Suicide intent – conscious desire to end one's life, indicating a serious mental health concern, often overwhelming emotional pain.  
辛: Worthlessness – a profound sense of having little or no value, feeling inadequate or undeserving.

Instructions:
1. Read the given post text carefully.
2. Based on the content, identify all emotions from the list above that are present or implied in the post. Each post is guaranteed to contain at least one of these emotions.
3. Return your answer as a comma-separated list of the corresponding Chinese codes (e.g., "丙, 丁" for emptiness and hopelessness). Do not include any additional text or explanation.
4. Only output the list.

Post: {text}

Emotions expressed:"""

    response = client.chat.completions.create(
        model="gemini-3.1-pro-preview",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=3000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# ========== 初始化输出文件 ==========
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as f:
            existing_data = json.load(f)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件损坏或为空，重新初始化
        with open(output_json_path, "w") as f:
            json.dump([], f)
else:
    with open(output_json_path, "w") as f:
        json.dump([], f)

# ========== 重试处理 ==========
max_retries = 5
current_retry = 0
all_ids = set(data['id'].tolist())

while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    remaining_data = data[~data['id'].isin(success_ids)]

    if remaining_data.empty:
        break

    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = row['id']
            text = row['text']
            title = row.get('title', '')
            post = row.get('post', '')
            target = row['emotions']      # 原始情绪列表，如 ['hopelessness', 'sadness']
            label = row['label_id']       # 标签二进制字符串，如 "10100"

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查
            if Id in success_ids:
                continue

            # 调用模型预测
            predict = identify_emotions(text)

            # 构造结果对象
            result = {
                "Id": Id,
                "text": text,
                "title": title,
                "post": post,
                "predict": predict,
                "label": label,
                "target": target   # 保留原始情绪列表用于后续评估
            }

            # 实时写入文件
            with open(output_json_path, "r+") as f:
                file_data = json.load(f)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    f.seek(0)
                    json.dump(file_data, f, ensure_ascii=False, indent=4)
                    f.truncate()
                    success_ids.add(Id)
                else:
                    # 同步内存集合
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 避免请求过快

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            continue

# ========== 按原始顺序排序 ==========
print("正在按原始 CSV 顺序重新排序输出文件...")
id_to_order = {row['id']: idx for idx, row in data.iterrows()}

with open(output_json_path, "r") as f:
    final_data = json.load(f)

final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# ========== 最终统计 ==========
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

正在按原始 CSV 顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [4]:
import json
import numpy as np
from sklearn.metrics import (accuracy_score, hamming_loss, f1_score, 
                             precision_score, recall_score, 
                             precision_recall_fscore_support)

# ========== 配置 ==========
input_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/gemini-3.1-pro-preview_Depression_EIGHT.json"

# 情绪代码到英文名称的映射（顺序固定，用于二进制向量）
emotion_codes = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛']
emotion_names = [
    'anger',
    'brain dysfunction (forget)',
    'emptiness',
    'hopelessness',
    'loneliness',
    'sadness',
    'suicide intent',
    'worthlessness'
]
code_to_name = dict(zip(emotion_codes, emotion_names))

# ========== 读取数据 ==========
with open(input_json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# ========== 解析预测和真实标签 ==========
y_true = []   # 真实二进制向量
y_pred = []   # 预测二进制向量

for item in data:
    # 真实情绪列表
    true_emotions = set(item['target'])  # 示例: ["hopelessness", "sadness"]
    # 转换为二进制向量
    true_vec = [1 if name in true_emotions else 0 for name in emotion_names]
    # print(true_vec)
    y_true.append(true_vec)

    # 预测字符串
    pred_str = item.get('predict', '')
    # 提取出现的情绪代码
    pred_codes = {code for code in emotion_codes if code in pred_str}
    # print(pred_codes)
    # 转换为英文名称（用于统一表示，也可直接用代码构建向量）
    pred_emotions = {code_to_name[code] for code in pred_codes}
    # print(pred_emotions)
    pred_vec = [1 if name in pred_emotions else 0 for name in emotion_names]
    # print(pred_vec)
    y_pred.append(pred_vec)

# 转换为 numpy 数组
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ========== 计算指标 ==========
# 1. Exact Match Ratio (准确率)
exact_match = accuracy_score(y_true, y_pred)
print(f"Exact Match Ratio (准确率): {exact_match:.4f}")

# 2. Hamming Loss
h_loss = hamming_loss(y_true, y_pred)
print(f"Hamming Loss: {h_loss:.4f}")

# 3. 宏平均指标
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1-score:  {f1_macro:.4f}")

# 4. 微平均指标
precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
print(f"\nMicro Precision: {precision_micro:.4f}")
print(f"Micro Recall:    {recall_micro:.4f}")
print(f"Micro F1-score:  {f1_micro:.4f}")

# 5. 每个类别的精确率、召回率、F1-score
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0, labels=range(8)
)

print("\n每个类别的指标：")
for i, name in enumerate(emotion_names):
    print(f"{name}:")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-score:  {f1[i]:.4f}")
    print(f"  Support:   {support[i]}")

# 可选：输出总体统计
print(f"\n总样本数: {len(data)}")

Exact Match Ratio (准确率): 0.1104
Hamming Loss: 0.2841

Macro Precision: 0.7498
Macro Recall:    0.5142
Macro F1-score:  0.5952

Micro Precision: 0.7832
Micro Recall:    0.5131
Micro F1-score:  0.6200

每个类别的指标：
anger:
  Precision: 0.7237
  Recall:    0.6377
  F1-score:  0.6780
  Support:   345
brain dysfunction (forget):
  Precision: 0.5775
  Recall:    0.2398
  F1-score:  0.3388
  Support:   171
emptiness:
  Precision: 0.6510
  Recall:    0.3666
  F1-score:  0.4690
  Support:   341
hopelessness:
  Precision: 0.9046
  Recall:    0.4637
  F1-score:  0.6131
  Support:   634
loneliness:
  Precision: 0.7896
  Recall:    0.7595
  F1-score:  0.7743
  Support:   420
sadness:
  Precision: 0.9272
  Recall:    0.4749
  F1-score:  0.6281
  Support:   697
suicide intent:
  Precision: 0.7552
  Recall:    0.7845
  F1-score:  0.7696
  Support:   232
worthlessness:
  Precision: 0.6693
  Recall:    0.3871
  F1-score:  0.4905
  Support:   434

总样本数: 906


### Kimi-K2.5

In [ ]:
import pandas as pd
import json
import os
import time
from tqdm import tqdm
from openai import OpenAI

# ========== 配置 ==========
key = " "
input_json_path = "/data/zhengtianlong/Private/Depressed/DepressionEmo-main/Dataset/test.json"
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/kimi-k2.5_Depression_EIGHT.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# ========== 读取数据 ==========
data = pd.read_json(input_json_path, lines=True)  # JSON lines 格式
# 确保 id 列为字符串类型
data['id'] = data['id'].astype(str)

# ========== 定义情绪分类函数 ==========
def identify_emotions(text):
    """调用大模型，返回帖子中包含的情绪代码（如 '丙, 丁'）"""
    client = OpenAI(
        base_url="https://api.linkapi.org/v1",
        api_key=key
    )
    prompt = f"""You are an expert in mental health analysis. Your task is to analyze a social media post and identify which of the following depression-related emotions are expressed in it. Each emotion is represented by a Chinese character code (甲, 乙, 丙, 丁, 戊, 己, 庚, 辛) with corresponding descriptions:

甲: Anger – an intensified emotional response that can be directed inward or outward, contributing to despair and self-criticism.  
乙: Cognitive dysfunction – impairment in normal cognitive functioning, causing forgetfulness, slow thinking, difficulty expressing thoughts.  
丙: Emptiness – a deep emotional void, numbness, lack of vitality, feeling disconnected and an empty future.  
丁: Hopelessness – a susceptibility factor in depressive symptoms, intrinsic to depression.  
戊: Loneliness – profound isolation even in the presence of others, predisposing to depression.  
己: Sadness – a natural emotion triggered by events or losses, a fundamental symptom of depression.  
庚: Suicide intent – conscious desire to end one's life, indicating a serious mental health concern, often overwhelming emotional pain.  
辛: Worthlessness – a profound sense of having little or no value, feeling inadequate or undeserving.

Instructions:
1. Read the given post text carefully.
2. Based on the content, identify all emotions from the list above that are present or implied in the post. Each post is guaranteed to contain at least one of these emotions.
3. Return your answer as a comma-separated list of the corresponding Chinese codes (e.g., "丙, 丁" for emptiness and hopelessness). Do not include any additional text or explanation.
4. Only output the list.

Post: {text}

Emotions expressed:"""

    response = client.chat.completions.create(
        model="kimi-k2.5",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=3000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# ========== 初始化输出文件 ==========
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as f:
            existing_data = json.load(f)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件损坏或为空，重新初始化
        with open(output_json_path, "w") as f:
            json.dump([], f)
else:
    with open(output_json_path, "w") as f:
        json.dump([], f)

# ========== 重试处理 ==========
max_retries = 5
current_retry = 0
all_ids = set(data['id'].tolist())

while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    remaining_data = data[~data['id'].isin(success_ids)]

    if remaining_data.empty:
        break

    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = row['id']
            text = row['text']
            title = row.get('title', '')
            post = row.get('post', '')
            target = row['emotions']      # 原始情绪列表，如 ['hopelessness', 'sadness']
            label = row['label_id']       # 标签二进制字符串，如 "10100"

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查
            if Id in success_ids:
                continue

            # 调用模型预测
            predict = identify_emotions(text)

            # 构造结果对象
            result = {
                "Id": Id,
                "text": text,
                "title": title,
                "post": post,
                "predict": predict,
                "label": label,
                "target": target   # 保留原始情绪列表用于后续评估
            }

            # 实时写入文件
            with open(output_json_path, "r+") as f:
                file_data = json.load(f)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    f.seek(0)
                    json.dump(file_data, f, ensure_ascii=False, indent=4)
                    f.truncate()
                    success_ids.add(Id)
                else:
                    # 同步内存集合
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 避免请求过快

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            continue

# ========== 按原始顺序排序 ==========
print("正在按原始 CSV 顺序重新排序输出文件...")
id_to_order = {row['id']: idx for idx, row in data.iterrows()}

with open(output_json_path, "r") as f:
    final_data = json.load(f)

final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# ========== 最终统计 ==========
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

正在按原始 CSV 顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [2]:
import json
import numpy as np
from sklearn.metrics import (accuracy_score, hamming_loss, f1_score, 
                             precision_score, recall_score, 
                             precision_recall_fscore_support)

# ========== 配置 ==========
input_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/kimi-k2.5_Depression_EIGHT.json"

# 情绪代码到英文名称的映射（顺序固定，用于二进制向量）
emotion_codes = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛']
emotion_names = [
    'anger',
    'brain dysfunction (forget)',
    'emptiness',
    'hopelessness',
    'loneliness',
    'sadness',
    'suicide intent',
    'worthlessness'
]
code_to_name = dict(zip(emotion_codes, emotion_names))

# ========== 读取数据 ==========
with open(input_json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# ========== 解析预测和真实标签 ==========
y_true = []   # 真实二进制向量
y_pred = []   # 预测二进制向量

for item in data:
    # 真实情绪列表
    true_emotions = set(item['target'])  # 示例: ["hopelessness", "sadness"]
    # 转换为二进制向量
    true_vec = [1 if name in true_emotions else 0 for name in emotion_names]
    # print(true_vec)
    y_true.append(true_vec)

    # 预测字符串
    pred_str = item.get('predict', '')
    # 提取出现的情绪代码
    pred_codes = {code for code in emotion_codes if code in pred_str}
    # print(pred_codes)
    # 转换为英文名称（用于统一表示，也可直接用代码构建向量）
    pred_emotions = {code_to_name[code] for code in pred_codes}
    # print(pred_emotions)
    pred_vec = [1 if name in pred_emotions else 0 for name in emotion_names]
    # print(pred_vec)
    y_pred.append(pred_vec)

# 转换为 numpy 数组
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ========== 计算指标 ==========
# 1. Exact Match Ratio (准确率)
exact_match = accuracy_score(y_true, y_pred)
print(f"Exact Match Ratio (准确率): {exact_match:.4f}")

# 2. Hamming Loss
h_loss = hamming_loss(y_true, y_pred)
print(f"Hamming Loss: {h_loss:.4f}")

# 3. 宏平均指标
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1-score:  {f1_macro:.4f}")

# 4. 微平均指标
precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
print(f"\nMicro Precision: {precision_micro:.4f}")
print(f"Micro Recall:    {recall_micro:.4f}")
print(f"Micro F1-score:  {f1_micro:.4f}")

# 5. 每个类别的精确率、召回率、F1-score
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0, labels=range(8)
)

print("\n每个类别的指标：")
for i, name in enumerate(emotion_names):
    print(f"{name}:")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-score:  {f1[i]:.4f}")
    print(f"  Support:   {support[i]}")

# 可选：输出总体统计
print(f"\n总样本数: {len(data)}")

Exact Match Ratio (准确率): 0.0552
Hamming Loss: 0.3016

Macro Precision: 0.6791
Macro Recall:    0.6295
Macro F1-score:  0.6193

Micro Precision: 0.6668
Micro Recall:    0.6643
Micro F1-score:  0.6655

每个类别的指标：
anger:
  Precision: 0.7833
  Recall:    0.4609
  F1-score:  0.5803
  Support:   345
brain dysfunction (forget):
  Precision: 0.6327
  Recall:    0.1813
  F1-score:  0.2818
  Support:   171
emptiness:
  Precision: 0.4355
  Recall:    0.8123
  F1-score:  0.5670
  Support:   341
hopelessness:
  Precision: 0.8147
  Recall:    0.6104
  F1-score:  0.6979
  Support:   634
loneliness:
  Precision: 0.5754
  Recall:    0.8452
  F1-score:  0.6847
  Support:   420
sadness:
  Precision: 0.8100
  Recall:    0.8135
  F1-score:  0.8117
  Support:   697
suicide intent:
  Precision: 0.7452
  Recall:    0.8448
  F1-score:  0.7919
  Support:   232
worthlessness:
  Precision: 0.6364
  Recall:    0.4677
  F1-score:  0.5392
  Support:   434

总样本数: 906


### Qwen3.5-Plus

In [ ]:
import pandas as pd
import json
import os
import time
from tqdm import tqdm
from openai import OpenAI

# ========== 配置 ==========
key = " "
input_json_path = "/data/zhengtianlong/Private/Depressed/DepressionEmo-main/Dataset/test.json"
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/qwen3.5-plus_Depression_EIGHT.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# ========== 读取数据 ==========
data = pd.read_json(input_json_path, lines=True)  # JSON lines 格式
# 确保 id 列为字符串类型
data['id'] = data['id'].astype(str)

# ========== 定义情绪分类函数 ==========
def identify_emotions(text):
    """调用大模型，返回帖子中包含的情绪代码（如 '丙, 丁'）"""
    client = OpenAI(
        base_url="https://api.linkapi.org/v1",
        api_key=key
    )
    prompt = f"""You are an expert in mental health analysis. Your task is to analyze a social media post and identify which of the following depression-related emotions are expressed in it. Each emotion is represented by a Chinese character code (甲, 乙, 丙, 丁, 戊, 己, 庚, 辛) with corresponding descriptions:

甲: Anger – an intensified emotional response that can be directed inward or outward, contributing to despair and self-criticism.  
乙: Cognitive dysfunction – impairment in normal cognitive functioning, causing forgetfulness, slow thinking, difficulty expressing thoughts.  
丙: Emptiness – a deep emotional void, numbness, lack of vitality, feeling disconnected and an empty future.  
丁: Hopelessness – a susceptibility factor in depressive symptoms, intrinsic to depression.  
戊: Loneliness – profound isolation even in the presence of others, predisposing to depression.  
己: Sadness – a natural emotion triggered by events or losses, a fundamental symptom of depression.  
庚: Suicide intent – conscious desire to end one's life, indicating a serious mental health concern, often overwhelming emotional pain.  
辛: Worthlessness – a profound sense of having little or no value, feeling inadequate or undeserving.

Instructions:
1. Read the given post text carefully.
2. Based on the content, identify all emotions from the list above that are present or implied in the post. Each post is guaranteed to contain at least one of these emotions.
3. Return your answer as a comma-separated list of the corresponding Chinese codes (e.g., "丙, 丁" for emptiness and hopelessness). Do not include any additional text or explanation.
4. Only output the list.

Post: {text}

Emotions expressed:"""

    response = client.chat.completions.create(
        model="qwen3.5-plus",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=8000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# ========== 初始化输出文件 ==========
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as f:
            existing_data = json.load(f)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件损坏或为空，重新初始化
        with open(output_json_path, "w") as f:
            json.dump([], f)
else:
    with open(output_json_path, "w") as f:
        json.dump([], f)

# ========== 重试处理 ==========
max_retries = 5
current_retry = 0
all_ids = set(data['id'].tolist())

while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    remaining_data = data[~data['id'].isin(success_ids)]

    if remaining_data.empty:
        break

    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = row['id']
            text = row['text']
            title = row.get('title', '')
            post = row.get('post', '')
            target = row['emotions']      # 原始情绪列表，如 ['hopelessness', 'sadness']
            label = row['label_id']       # 标签二进制字符串，如 "10100"

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查
            if Id in success_ids:
                continue

            # 调用模型预测
            predict = identify_emotions(text)

            # 构造结果对象
            result = {
                "Id": Id,
                "text": text,
                "title": title,
                "post": post,
                "predict": predict,
                "label": label,
                "target": target   # 保留原始情绪列表用于后续评估
            }

            # 实时写入文件
            with open(output_json_path, "r+") as f:
                file_data = json.load(f)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    f.seek(0)
                    json.dump(file_data, f, ensure_ascii=False, indent=4)
                    f.truncate()
                    success_ids.add(Id)
                else:
                    # 同步内存集合
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 避免请求过快

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            continue

# ========== 按原始顺序排序 ==========
print("正在按原始 CSV 顺序重新排序输出文件...")
id_to_order = {row['id']: idx for idx, row in data.iterrows()}

with open(output_json_path, "r") as f:
    final_data = json.load(f)

final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# ========== 最终统计 ==========
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

In [3]:
import json
import numpy as np
from sklearn.metrics import (accuracy_score, hamming_loss, f1_score, 
                             precision_score, recall_score, 
                             precision_recall_fscore_support)

# ========== 配置 ==========
input_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/qwen3.5-plus_Depression_EIGHT.json"

# 情绪代码到英文名称的映射（顺序固定，用于二进制向量）
emotion_codes = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛']
emotion_names = [
    'anger',
    'brain dysfunction (forget)',
    'emptiness',
    'hopelessness',
    'loneliness',
    'sadness',
    'suicide intent',
    'worthlessness'
]
code_to_name = dict(zip(emotion_codes, emotion_names))

# ========== 读取数据 ==========
with open(input_json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# ========== 解析预测和真实标签 ==========
y_true = []   # 真实二进制向量
y_pred = []   # 预测二进制向量

for item in data:
    # 真实情绪列表
    true_emotions = set(item['target'])  # 示例: ["hopelessness", "sadness"]
    # 转换为二进制向量
    true_vec = [1 if name in true_emotions else 0 for name in emotion_names]
    # print(true_vec)
    y_true.append(true_vec)

    # 预测字符串
    pred_str = item.get('predict', '')
    # 提取出现的情绪代码
    pred_codes = {code for code in emotion_codes if code in pred_str}
    # print(pred_codes)
    # 转换为英文名称（用于统一表示，也可直接用代码构建向量）
    pred_emotions = {code_to_name[code] for code in pred_codes}
    # print(pred_emotions)
    pred_vec = [1 if name in pred_emotions else 0 for name in emotion_names]
    # print(pred_vec)
    y_pred.append(pred_vec)

# 转换为 numpy 数组
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ========== 计算指标 ==========
# 1. Exact Match Ratio (准确率)
exact_match = accuracy_score(y_true, y_pred)
print(f"Exact Match Ratio (准确率): {exact_match:.4f}")

# 2. Hamming Loss
h_loss = hamming_loss(y_true, y_pred)
print(f"Hamming Loss: {h_loss:.4f}")

# 3. 宏平均指标
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1-score:  {f1_macro:.4f}")

# 4. 微平均指标
precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
print(f"\nMicro Precision: {precision_micro:.4f}")
print(f"Micro Recall:    {recall_micro:.4f}")
print(f"Micro F1-score:  {f1_micro:.4f}")

# 5. 每个类别的精确率、召回率、F1-score
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0, labels=range(8)
)

print("\n每个类别的指标：")
for i, name in enumerate(emotion_names):
    print(f"{name}:")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-score:  {f1[i]:.4f}")
    print(f"  Support:   {support[i]}")

# 可选：输出总体统计
print(f"\n总样本数: {len(data)}")

Exact Match Ratio (准确率): 0.0928
Hamming Loss: 0.2749

Macro Precision: 0.7314
Macro Recall:    0.5443
Macro F1-score:  0.6122

Micro Precision: 0.7655
Micro Recall:    0.5641
Micro F1-score:  0.6495

每个类别的指标：
anger:
  Precision: 0.7077
  Recall:    0.5843
  F1-score:  0.6401
  Support:   344
brain dysfunction (forget):
  Precision: 0.5789
  Recall:    0.1941
  F1-score:  0.2907
  Support:   170
emptiness:
  Precision: 0.6650
  Recall:    0.3959
  F1-score:  0.4963
  Support:   341
hopelessness:
  Precision: 0.8571
  Recall:    0.6066
  F1-score:  0.7105
  Support:   633
loneliness:
  Precision: 0.7604
  Recall:    0.7405
  F1-score:  0.7503
  Support:   420
sadness:
  Precision: 0.8733
  Recall:    0.5546
  F1-score:  0.6784
  Support:   696
suicide intent:
  Precision: 0.7603
  Recall:    0.7931
  F1-score:  0.7764
  Support:   232
worthlessness:
  Precision: 0.6481
  Recall:    0.4850
  F1-score:  0.5548
  Support:   433

总样本数: 905
